# SQL-Based Insight Validation

This notebook demonstrates automated cross-layer metric validation between SQL views and Python calculations:
1. **Task 1 & 2: Computing & Comparing 3 Metrics** (Active Users, AOV, Monthly Churn).
2. **Task 3: Running Automated `validate_metrics()` Script**.
3. **Task 4: Investigating Discrepancies & Hand-Computing Sample Subsets**.
4. **Task 5: Answering Follow-Up Questions on Manual Review & Auto-Fix Risks**.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

db_path = '../analytics.db'
if not os.path.exists(db_path):
    db_path = 'analytics.db'

engine = create_engine(f'sqlite:///{db_path}')
print(f"Connected to validation database: {db_path}")

## Task 1 & 2: Cross-Layer Metric Audit Results

In [2]:
report_path = 'validation_report.csv' if os.path.exists('validation_report.csv') else '../validation_report.csv'
report_df = pd.read_csv(report_path)
print("Validation Audit Report (Post-Fix):")
print(report_df)

## Task 4: Sample Subset Hand-Computation Verification

In [3]:
orders_df = pd.read_sql("SELECT customer_id, order_date, order_amount FROM orders", engine)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

now = pd.Timestamp.now()
m_n_start = now.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
m_n1_start = (m_n_start - pd.DateOffset(months=1)).replace(day=1)

sample_orders = orders_df[(orders_df['customer_id'] >= 1001) & (orders_df['customer_id'] <= 1050)]
cust_n1 = set(sample_orders[(sample_orders['order_date'] >= m_n1_start) & (sample_orders['order_date'] < m_n_start) & (sample_orders['order_amount'] > 0)]['customer_id'])
cust_n = set(sample_orders[(sample_orders['order_date'] >= m_n_start) & (sample_orders['order_amount'] > 0)]['customer_id'])

sample_churn = cust_n1 - cust_n
print(f"Hand-Computed Sample Subset Churn (Customers 1001-1050): {len(sample_churn)} customers")
print(f"Churned Customer IDs: {sorted(list(sample_churn))}")